# 03 ベースライン

`01_data_fetch` が作った `data/processed/observations.parquet`（1 行 ＝ 都市 × 正時）に目的変数を付け、
概算式 `125 × (気温 − 露点)` [m] の誤差（MAE）を、都市別・季節別に出して `reports/baseline.json` に保存する。

- 目的変数・分割・概算式・評価関数の中身は `ml/` にある（テスト済み）。このノートブックは順に呼ぶだけ。
- 目的変数（`docs/spec.md`「9. 目的変数」）：最も低い BKN・OVC・`VV` の高さを、上限 1,524 m（5,000 フィート）で打ち切った値。
  雲が少ない・ない正時は 1,524 m。正解の分からない正時（高さのない BKN・OVC・`VV` だけ、METAR なし）は除く。
- **評価するのは学習（2018〜2023）と検証（2024）の年だけ**。テストの年（2025）の誤差は、ここでは出さない
  （成功基準を決める前に見ないため。`05_evaluate` で同じ関数を使って出す）。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。動くことの確認用で、数値に意味はない。一時的な場所に書く）。
  `reports/baseline.json` は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。
- 時刻は UTC（分割の年も UTC）。季節だけ現地の月。単位は摂氏・メートル。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import archive, baseline, dataset, eda, evaluate, features, metar, raw_store, split, target

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    OUTPUT = ROOT / "reports" / "baseline.json"
    # テストの年は評価しない。
    SPLITS = {name: list(split.SPLIT_YEARS[name]) for name in (split.TRAIN, split.VALIDATION)}
elif MODE == "sample":
    OUTPUT = Path(tempfile.mkdtemp()) / "baseline.json"
    # fixture は 2025 年の 2 日ぶんだけなので、分割で絞らずに 1 つの組として評価する。
    SPLITS = {"sample": [2025]}
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"mode={MODE}", list(SPLITS))

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)

dataset.validate_dataset(observations)
print(observations.shape, observations["time"].min(), observations["time"].max())

## 1. 目的変数

`target_m` が欠損の行は、学習・評価から除く行。件数（行の数）だけは全期間で数える（`docs/eda.md` の件数と突き合わせるため）。

In [ ]:
observations["target_m"] = target.target_m(observations)
observations["status"] = eda.ceiling_status(observations)
observations["season"] = features.season(observations)
observations["split"] = split.split(observations) if MODE == "full" else "sample"

observations["kind"] = "below_cap"
observations.loc[observations["target_m"] >= target.CEILING_CAP_M, "kind"] = "cap"
observations.loc[observations["target_m"].isna(), "kind"] = "excluded"

pd.crosstab([observations["city"], observations["split"]], observations["kind"], margins=True)

In [ ]:
# 除いた行の内訳（全期間）。docs/eda.md の件数：height_unknown は東京 11,397・ベルリン 16、
# no_metar は 13、VV/// は 68。
excluded = observations[observations["target_m"].isna()]
excluded_counts = pd.crosstab(excluded["city"], excluded["status"])
excluded_counts

In [ ]:
# `VV` と BKN・OVC が同じ行にある件数（目的変数の定義と `eda.ceiling_status` で扱いが違う行）。
covers = observations[[f"sky_cover_{layer}" for layer in metar.LAYERS]].apply(
    lambda column: column.str.strip()
)
has_vertical_visibility = covers.isin(["VV"]).any(axis=1)
has_bkn_or_ovc = covers.isin(metar.CEILING_COVERS).any(axis=1)
vertical_visibility_with_bkn_or_ovc = int((has_vertical_visibility & has_bkn_or_ovc).sum())
print("VV:", int(has_vertical_visibility.sum()))
print("VV with BKN/OVC:", vertical_visibility_with_bkn_or_ovc)

## 2. ベースライン

- `rule_of_thumb`：**正式なベースライン**。Open-Meteo の気温・露点で計算した概算式を、0〜1,524 m に収めた値（API の `baseline_m` と同じ入力）。
- `always_cap`：参考。いつも上限（1,524 m）と答える。全体の MAE が、上限の行の多さでどこまで決まるかを見る。
- `rule_of_thumb_metar`：参考。METAR の気温・露点で計算した概算式。式の限界か、入力の粗さかを見る。
  METAR に気温・露点のない行は評価から除く（件数を出す）。

In [ ]:
predictions = {
    "rule_of_thumb": baseline.baseline_m(
        observations["temperature_c"], observations["dew_point_c"]
    ),
    "always_cap": pd.Series(target.CEILING_CAP_M, index=observations.index, dtype="float64"),
    "rule_of_thumb_metar": baseline.baseline_m(
        observations["metar_temperature_c"], observations["metar_dew_point_c"]
    ),
}

spread = features.dew_point_spread_c(observations["temperature_c"], observations["dew_point_c"])
has_target = observations["target_m"].notna()
print("negative spread (Open-Meteo):", int((spread < 0).sum()))
print(
    "rows with target but without METAR temperature or dew point:",
    int((has_target & predictions["rule_of_thumb_metar"].isna()).sum()),
)
pd.DataFrame(predictions).describe().T

## 3. 評価

In [ ]:
results = {}
for name, years in SPLITS.items():
    rows = observations["split"] == name
    part = observations[rows]
    results[name] = {
        "years": years,
        "rows": {
            city: {kind: int(count) for kind, count in counts.items()}
            for city, counts in pd.crosstab(part["city"], part["kind"]).iterrows()
        },
        "predictions": {},
    }
    for label, prediction in predictions.items():
        truth = part["target_m"]
        if label == "rule_of_thumb_metar":
            # 参考の予測だけ、予測を作れない行を評価から除く。
            truth = truth.where(prediction[rows].notna())
        results[name]["predictions"][label] = evaluate.report(part, truth, prediction[rows])

list(results)

In [ ]:
by_city = pd.DataFrame(
    [
        {"split": name, "prediction": label, "city": city, **metrics}
        for name, result in results.items()
        for label, reports in result["predictions"].items()
        for city, metrics in reports["by_city"].items()
    ]
)
by_city

In [ ]:
by_season = pd.DataFrame(
    [
        {"split": name, "city": city, "season": season, **metrics}
        for name, result in results.items()
        for city, seasons in result["predictions"]["rule_of_thumb"]["by_city_season"].items()
        for season, metrics in seasons.items()
    ]
)
by_season

## 4. 保存

実行した時刻は入れない（再実行しても同じファイルになる）。

In [ ]:
report = {
    "mode": MODE,
    "cap_m": target.CEILING_CAP_M,
    "formula": "125 * (temperature_c - dew_point_c), clipped to [0, cap_m]",
    "predictions": {
        "rule_of_thumb": "baseline: formula with Open-Meteo temperature and dew point",
        "always_cap": "reference: always cap_m",
        "rule_of_thumb_metar": "reference: formula with METAR temperature and dew point",
    },
    "excluded_rows_all_years": {
        city: {status: int(count) for status, count in counts.items() if count}
        for city, counts in excluded_counts.iterrows()
    },
    "vertical_visibility_with_bkn_or_ovc_all_years": vertical_visibility_with_bkn_or_ovc,
    "splits": results,
}

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
text = json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
OUTPUT.write_text(text, encoding="utf-8", newline="\n")
assert json.loads(OUTPUT.read_text(encoding="utf-8")) == report
print(OUTPUT.name, len(text))